# Training Models: Full 2010–2021 10-fold OOF Benchmark

## 1. Objective and scope

This is the production training-stage Notebook for the four submitted Steam negative-review classifiers. **Run All defaults to the complete eligible 2010–2021 cohort and exactly 10 shared folds.** It does not inspect 2022/2023 outcomes, calibrate probabilities, or select a final threshold.

The prior audit is preserved separately. This Notebook performs only the minimal safety checks required before training and uses disk-backed fold partitions so the full feature union never remains resident in memory.

## 2. Reproducibility configuration

Default behavior is `full`. For development, development runs can set environment overrides and write to a temporary directory. Full-training refits remain disabled unless `TRAINING_CV_RUN_FULL_REFITS=1` is explicitly set.

In [ ]:
import gc
import json
import os
import platform
import random
try:
    import resource
except ImportError:
    resource = None
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path((os.environ.get("TEAM18_PROJECT_ROOT") or next(
    str(p) for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "run.py").is_file() and (p / "src" / "steam_reviews").is_dir()
))).resolve()
MODELS_DIR = PROJECT_ROOT / "src"
if str(MODELS_DIR) not in sys.path:
    sys.path.insert(0, str(MODELS_DIR))

os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "runs" / "matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import pyarrow.dataset as ds
import seaborn as sns
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler
from threadpoolctl import threadpool_limits

from steam_reviews.modeling.training_cv_disk import (
    DATE_COLUMN,
    TARGET,
    TRAIN_END,
    assert_resources_safe,
    build_or_reuse_staging,
    estimate_full_resources,
    expected_checkpoint_signature,
    load_model_fold_data,
    load_run_config,
    sha256_file,
    staging_disk_bytes,
    stable_json_sha256,
    validate_checkpoint,
)

config = load_run_config(PROJECT_ROOT)
RUN_MODE = config.run_mode
RANDOM_SEED = config.random_seed
N_OUTER_FOLDS = config.n_outer_folds
RUN_FULL_REFITS = config.run_full_refits
FULL_PREFLIGHT_ONLY = config.full_preflight_only
ARTIFACT_DIR = config.artifact_dir
FIGURE_DIR = ARTIFACT_DIR / "figures"
CHECKPOINT_DIR = config.checkpoint_dir
FITTED_DIR = ARTIFACT_DIR / "fitted_models"
SUFFIX = "" if RUN_MODE == "full" else "_smoke"
for directory in (ARTIFACT_DIR, FIGURE_DIR, CHECKPOINT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_rows", 30)
pd.set_option("display.max_columns", 30)

def peak_rss_gb():
    if resource is None:
        return psutil.Process().memory_info().rss / 1024**3
    value = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return value / 1024**3 if sys.platform == "darwin" else value / 1024**2

run_started = time.perf_counter()
display(pd.Series({
    "run_mode": RUN_MODE,
    "folds": N_OUTER_FOLDS,
    "source": str(config.source_path),
    "artifacts": str(ARTIFACT_DIR),
    "full_preflight_only": FULL_PREFLIGHT_ONLY,
    "run_full_refits": RUN_FULL_REFITS,
}, name="Configuration").to_frame("value"))

## 3. Completed full-run source contract

The archived full-run source hash is verified before training. Development smoke results and old standalone model scripts are not required by this submission copy. An explicit source override remains available for development.

In [ ]:
# Verify the original completed full-run source, independently of omitted development files.
PRODUCTION_AUDIT_DIR = PROJECT_ROOT / "results" / "cross_validation"
audited_run = json.loads((PRODUCTION_AUDIT_DIR / "run_manifest.json").read_text())
data_path_overridden = "TRAINING_CV_DATA_PATH" in os.environ
EXPECTED_SOURCE_SHA256 = None if data_path_overridden else audited_run["source_sha256"]
source_hash_before = sha256_file(config.source_path)
if EXPECTED_SOURCE_SHA256 is not None:
    assert source_hash_before == EXPECTED_SOURCE_SHA256
print("Completed full-run source hash verified.")


## 4. Data discovery and schema

Only schema metadata is inspected here. The staging helper projects the minimum cohort columns on pass one and only the approved predictor union on pass two.

In [ ]:
source_dataset = ds.dataset(config.source_path, format="parquet")
source_columns = source_dataset.schema.names
genre_features = ['genre_Action', 'genre_Adventure', 'genre_Arcade', 'genre_Building', 'genre_Card & Board Game', 'genre_Casual', 'genre_Combat', 'genre_Comedy', 'genre_Detective-mystery', 'genre_Early Access', 'genre_Exploration', 'genre_FPP', 'genre_Fantasy', 'genre_Fighting', 'genre_Free to Play', 'genre_Gore', "genre_Hack and slash/Beat 'em up", 'genre_Historical', 'genre_Horror', 'genre_Indie', 'genre_JRPG', 'genre_Managerial', 'genre_Massively Multiplayer', 'genre_Modern', 'genre_Music', 'genre_Mystery', 'genre_Narrative', 'genre_Nudity', 'genre_Open World', 'genre_Pinball', 'genre_Platformer', 'genre_Point-and-click', 'genre_Puzzle', 'genre_Quiz/Trivia', 'genre_Racing', 'genre_Real-time', 'genre_Roguelike', 'genre_Role-playing (RPG)', 'genre_Sci-fi', 'genre_Sexual Content', 'genre_Shooter', 'genre_Simulation', 'genre_Sports', 'genre_Strategy', 'genre_Survival', 'genre_TPP', 'genre_Tactical', 'genre_Turn-based', 'genre_Violent', 'genre_Visual Novel', 'genre_other']
p_features = ['p_tech_performance', 'p_tech_stability', 'p_tech_functionality', 'p_tech_save_integrity', 'p_tech_network_responsiveness', 'p_tech_network_session_access', 'p_tech_platform_compatibility', 'p_gameplay_core_loop', 'p_gameplay_balance', 'p_gameplay_challenge', 'p_gameplay_randomness', 'p_progression_effort_reward', 'p_content_scope_variety', 'p_content_narrative', 'p_content_world_level', 'p_social_cheating', 'p_social_skill_matchmaking', 'p_social_match_availability', 'p_social_conduct', 'p_social_sanctions', 'p_ux_interface', 'p_ux_controls', 'p_ux_onboarding', 'p_ux_localization', 'p_ux_accessibility', 'p_business_price_value', 'p_business_paid_advantage', 'p_business_paywall', 'p_business_sales_pressure', 'p_business_account_security', 'p_service_maintenance', 'p_service_marketing_trust', 'p_service_support', 'p_service_release_readiness', 'p_service_update_regression']
flag_features = ['flag_tech_performance', 'flag_tech_stability', 'flag_tech_functionality', 'flag_tech_save_integrity', 'flag_tech_network_responsiveness', 'flag_tech_network_session_access', 'flag_tech_platform_compatibility', 'flag_gameplay_core_loop', 'flag_gameplay_balance', 'flag_gameplay_challenge', 'flag_gameplay_randomness', 'flag_progression_effort_reward', 'flag_content_scope_variety', 'flag_content_narrative', 'flag_content_world_level', 'flag_social_cheating', 'flag_social_skill_matchmaking', 'flag_social_match_availability', 'flag_social_conduct', 'flag_social_sanctions', 'flag_ux_interface', 'flag_ux_controls', 'flag_ux_onboarding', 'flag_ux_localization', 'flag_ux_accessibility', 'flag_business_price_value', 'flag_business_paid_advantage', 'flag_business_paywall', 'flag_business_sales_pressure', 'flag_business_account_security', 'flag_service_maintenance', 'flag_service_marketing_trust', 'flag_service_support', 'flag_service_release_readiness', 'flag_service_update_regression']
assert len(genre_features) == 51
assert len(p_features) == 35
assert len(flag_features) == 35
print(f"Source schema: {len(source_columns)} columns")

## 5. Training-cohort construction

The full cohort is exact-binary `still_playing_d30`, unedited reviews, and dates before 2022. Full mode retains every eligible row; smoke mode takes a deterministic stratified subset. Rows are staged by shared fold on disk.

## 6. Target handling

Target-NULL causes were already audited and preserved in the earlier artifacts. This training-only Notebook does not rebuild that audit or fill NULL targets; NULL and non-binary rows are simply ineligible for supervised staging.

## 7. Leakage controls and feature groups

Feature lists preserve each submitted candidate. Identifiers, targets, snapshot variables, post-review engagement fields, votes/comments, app ID, and edited state are excluded from every predictor matrix.

In [ ]:
LEAKAGE_COLUMNS = {
    "recommendationid", "appid", "game", "review_updated_utc",
    "last_played_utc", "snapshot_date", "review_age_days_at_snapshot",
    "hours_forever_at_snapshot", "hours_last_2w_at_snapshot",
    "still_playing_d7", "still_playing_d30", "still_playing_d90",
    "still_playing_d180", "no_play_after_review", "days_from_review_to_last_play",
    "extra_hours_after_review_NOT_WINDOWED", "votes_up", "votes_funny",
    "weighted_vote_score", "comment_count", "edited", "num_games_owned",
    "author_num_reviews",
}
MODEL_FEATURES = {'logistic_structured': ['log1p_hours_at_review', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'genre_Action', 'genre_Adventure', 'genre_Casual', 'genre_Indie', 'genre_Massively Multiplayer', 'genre_Role-playing (RPG)', 'genre_Simulation', 'genre_Sports', 'genre_Strategy', 'genre_Shooter', 'genre_Puzzle', 'genre_Racing'], 'logistic_structured_nlp': ['hours_at_review', 'review_length_chars', 'language', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'genre_Action', 'genre_Adventure', 'genre_Arcade', 'genre_Building', 'genre_Card & Board Game', 'genre_Casual', 'genre_Combat', 'genre_Comedy', 'genre_Detective-mystery', 'genre_Early Access', 'genre_Exploration', 'genre_FPP', 'genre_Fantasy', 'genre_Fighting', 'genre_Free to Play', 'genre_Gore', "genre_Hack and slash/Beat 'em up", 'genre_Historical', 'genre_Horror', 'genre_Indie', 'genre_JRPG', 'genre_Managerial', 'genre_Massively Multiplayer', 'genre_Modern', 'genre_Music', 'genre_Mystery', 'genre_Narrative', 'genre_Nudity', 'genre_Open World', 'genre_Pinball', 'genre_Platformer', 'genre_Point-and-click', 'genre_Puzzle', 'genre_Quiz/Trivia', 'genre_Racing', 'genre_Real-time', 'genre_Roguelike', 'genre_Role-playing (RPG)', 'genre_Sci-fi', 'genre_Sexual Content', 'genre_Shooter', 'genre_Simulation', 'genre_Sports', 'genre_Strategy', 'genre_Survival', 'genre_TPP', 'genre_Tactical', 'genre_Turn-based', 'genre_Violent', 'genre_Visual Novel', 'genre_other', 'flag_tech_performance', 'flag_tech_stability', 'flag_tech_functionality', 'flag_tech_save_integrity', 'flag_tech_network_responsiveness', 'flag_tech_network_session_access', 'flag_tech_platform_compatibility', 'flag_gameplay_core_loop', 'flag_gameplay_balance', 'flag_gameplay_challenge', 'flag_gameplay_randomness', 'flag_progression_effort_reward', 'flag_content_scope_variety', 'flag_content_narrative', 'flag_content_world_level', 'flag_social_cheating', 'flag_social_skill_matchmaking', 'flag_social_match_availability', 'flag_social_conduct', 'flag_social_sanctions', 'flag_ux_interface', 'flag_ux_controls', 'flag_ux_onboarding', 'flag_ux_localization', 'flag_ux_accessibility', 'flag_business_price_value', 'flag_business_paid_advantage', 'flag_business_paywall', 'flag_business_sales_pressure', 'flag_business_account_security', 'flag_service_maintenance', 'flag_service_marketing_trust', 'flag_service_support', 'flag_service_release_readiness', 'flag_service_update_regression', 'n_factors_flagged'], 'lightgbm_structured': ['hours_at_review', 'review_length_chars', 'language', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'review_year', 'review_month', 'review_quarter_of_year', 'genre_Action', 'genre_Adventure', 'genre_Arcade', 'genre_Building', 'genre_Card & Board Game', 'genre_Casual', 'genre_Combat', 'genre_Comedy', 'genre_Detective-mystery', 'genre_Early Access', 'genre_Exploration', 'genre_FPP', 'genre_Fantasy', 'genre_Fighting', 'genre_Free to Play', 'genre_Gore', "genre_Hack and slash/Beat 'em up", 'genre_Historical', 'genre_Horror', 'genre_Indie', 'genre_JRPG', 'genre_Managerial', 'genre_Massively Multiplayer', 'genre_Modern', 'genre_Music', 'genre_Mystery', 'genre_Narrative', 'genre_Nudity', 'genre_Open World', 'genre_Pinball', 'genre_Platformer', 'genre_Point-and-click', 'genre_Puzzle', 'genre_Quiz/Trivia', 'genre_Racing', 'genre_Real-time', 'genre_Roguelike', 'genre_Role-playing (RPG)', 'genre_Sci-fi', 'genre_Sexual Content', 'genre_Shooter', 'genre_Simulation', 'genre_Sports', 'genre_Strategy', 'genre_Survival', 'genre_TPP', 'genre_Tactical', 'genre_Turn-based', 'genre_Violent', 'genre_Visual Novel', 'genre_other'], 'lightgbm_structured_nlp': ['log_hours_at_review', 'language', 'review_year', 'review_month', 'review_dayofweek', 'review_quarter_of_year', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'review_length_chars', 'genre_Action', 'genre_Adventure', 'genre_Arcade', 'genre_Building', 'genre_Card & Board Game', 'genre_Casual', 'genre_Combat', 'genre_Comedy', 'genre_Detective-mystery', 'genre_Early Access', 'genre_Exploration', 'genre_FPP', 'genre_Fantasy', 'genre_Fighting', 'genre_Free to Play', 'genre_Gore', "genre_Hack and slash/Beat 'em up", 'genre_Historical', 'genre_Horror', 'genre_Indie', 'genre_JRPG', 'genre_Managerial', 'genre_Massively Multiplayer', 'genre_Modern', 'genre_Music', 'genre_Mystery', 'genre_Narrative', 'genre_Nudity', 'genre_Open World', 'genre_Pinball', 'genre_Platformer', 'genre_Point-and-click', 'genre_Puzzle', 'genre_Quiz/Trivia', 'genre_Racing', 'genre_Real-time', 'genre_Roguelike', 'genre_Role-playing (RPG)', 'genre_Sci-fi', 'genre_Sexual Content', 'genre_Shooter', 'genre_Simulation', 'genre_Sports', 'genre_Strategy', 'genre_Survival', 'genre_TPP', 'genre_Tactical', 'genre_Turn-based', 'genre_Violent', 'genre_Visual Novel', 'genre_other', 'p_tech_performance', 'p_tech_stability', 'p_tech_functionality', 'p_tech_save_integrity', 'p_tech_network_responsiveness', 'p_tech_network_session_access', 'p_tech_platform_compatibility', 'p_gameplay_core_loop', 'p_gameplay_balance', 'p_gameplay_challenge', 'p_gameplay_randomness', 'p_progression_effort_reward', 'p_content_scope_variety', 'p_content_narrative', 'p_content_world_level', 'p_social_cheating', 'p_social_skill_matchmaking', 'p_social_match_availability', 'p_social_conduct', 'p_social_sanctions', 'p_ux_interface', 'p_ux_controls', 'p_ux_onboarding', 'p_ux_localization', 'p_ux_accessibility', 'p_business_price_value', 'p_business_paid_advantage', 'p_business_paywall', 'p_business_sales_pressure', 'p_business_account_security', 'p_service_maintenance', 'p_service_marketing_trust', 'p_service_support', 'p_service_release_readiness', 'p_service_update_regression']}
MODEL_FAMILY = {
    "logistic_structured": "Logistic Regression",
    "logistic_structured_nlp": "Logistic Regression",
    "lightgbm_structured": "LightGBM",
    "lightgbm_structured_nlp": "LightGBM",
}
MODEL_HAS_NLP = {
    "logistic_structured": False,
    "logistic_structured_nlp": True,
    "lightgbm_structured": False,
    "lightgbm_structured_nlp": True,
}
MODEL_PARAMETERS = {
    "logistic_structured": {"penalty": "none", "solver": "lbfgs", "max_iter": 2000, "tol": 1e-5},
    "logistic_structured_nlp": {"penalty": "l2", "solver": "lbfgs", "lambdas": [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6]},
    "lightgbm_structured": {"n_estimators": 200, "learning_rate": 0.05, "num_leaves": 15, "max_depth": 5, "min_child_samples": 30},
    "lightgbm_structured_nlp": {"max_estimators": 1000, "learning_rate": 0.05, "num_leaves": 31, "min_child_samples": 500, "reg_alpha": 0.1, "reg_lambda": 1.0},
}
for model_name, features in MODEL_FEATURES.items():
    assert len(features) == len(set(features))
    assert not (set(features) & LEAKAGE_COLUMNS)
display(pd.DataFrame({
    "model": list(MODEL_FEATURES),
    "feature_count": [len(MODEL_FEATURES[name]) for name in MODEL_FEATURES],
    "contains_nlp": [MODEL_HAS_NLP[name] for name in MODEL_FEATURES],
}))

## 8. Shared fold construction

The staging helper generates one deterministic fold assignment and writes Hive-style `fold_id=<n>` partitions. Every candidate reads the same saved folds.

In [ ]:
manifest = build_or_reuse_staging(
    config=config,
    source_path=config.source_path,
    model_features=MODEL_FEATURES,
    required_metadata=["recommendationid", DATE_COLUMN, TARGET, "edited"],
    expected_source_sha256=EXPECTED_SOURCE_SHA256,
)
fold_path = ARTIFACT_DIR / f"fold_assignments{SUFFIX}.parquet"
fold_assignments = pd.read_parquet(fold_path).sort_values("stable_row_id").reset_index(drop=True)
assert len(fold_assignments) == manifest.row_count
assert set(fold_assignments["fold_id"]) == set(range(N_OUTER_FOLDS))
assert fold_assignments["stable_row_id"].is_unique
assert fold_assignments["actual_label"].isin([0, 1]).all()
assert pd.to_datetime(fold_assignments[DATE_COLUMN], utc=True).max() < TRAIN_END
fold_class_balance = fold_assignments.groupby("fold_id")["actual_label"].agg(
    rows="size", positive_rows="sum", positive_rate="mean"
).reset_index()
display(fold_class_balance)

## 9. Model definitions

Preprocessing is fitted only on each outer-training fold. Logistic NLP keeps its submitted fold-internal lambda search. LightGBM NLP chooses early-stopping iterations on an inner split and then refits on the complete outer-training fold.

In [ ]:
LAMBDAS = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
N_INNER_FOLDS = 3 if RUN_MODE == "full" else 2

def log1p_clip(values):
    array = np.asarray(values, dtype="float32")
    return np.log1p(np.clip(array, 0, None)).astype("float32")

def logistic_structured_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("remove_constants", VarianceThreshold()),
        ("scale", StandardScaler(copy=False)),
        ("model", LogisticRegression(
            solver="lbfgs", C=np.inf, l1_ratio=0, random_state=RANDOM_SEED,
            max_iter=2000, tol=1e-5,
        )),
    ])

def logistic_nlp_pipeline():
    numeric_log = ["hours_at_review", "review_length_chars"]
    binary = [
        "steam_purchase", "received_for_free", "written_during_early_access"
    ] + genre_features + flag_features + ["n_factors_flagged"]
    prep = ColumnTransformer([
        ("numeric", Pipeline([
            ("fill", SimpleImputer(strategy="constant", fill_value=0)),
            ("log", FunctionTransformer(log1p_clip, feature_names_out="one-to-one")),
        ]), numeric_log),
        ("language", OneHotEncoder(
            max_categories=13, handle_unknown="infrequent_if_exist",
            sparse_output=False, dtype=np.float32,
        ), ["language"]),
        ("binary", SimpleImputer(strategy="constant", fill_value=0), binary),
    ], verbose_feature_names_out=False)
    return Pipeline([
        ("prep", prep),
        ("scale", StandardScaler(copy=False)),
        ("model", LogisticRegression(
            solver="lbfgs", l1_ratio=0, random_state=RANDOM_SEED, max_iter=1000,
        )),
    ])

def lightgbm_preprocessor(features):
    numeric = [feature for feature in features if feature != "language"]
    transformers = []
    if "language" in features:
        language = Pipeline([
            ("fill", SimpleImputer(strategy="most_frequent")),
            ("encode", OrdinalEncoder(
                handle_unknown="use_encoded_value", unknown_value=-1,
                encoded_missing_value=-1, dtype=np.float32,
            )),
        ])
        transformers.append(("language", language, ["language"]))
    transformers.append(("numeric", "passthrough", numeric))
    return ColumnTransformer(transformers, verbose_feature_names_out=False)

def probability_metrics(y_true, probability):
    return {
        "average_precision": float(average_precision_score(y_true, probability)),
        "roc_auc": float(roc_auc_score(y_true, probability)),
        "log_loss": float(log_loss(y_true, probability, labels=[0, 1])),
        "brier_score": float(brier_score_loss(y_true, probability)),
    }

def top_fraction_metrics(y_true, probability, fraction):
    count = max(1, int(np.ceil(len(y_true) * fraction)))
    order = np.argsort(-probability, kind="stable")[:count]
    positives = np.asarray(y_true)[order].sum()
    precision = positives / count
    recall = positives / np.asarray(y_true).sum()
    return {"precision": precision, "recall": recall, "lift": precision / np.asarray(y_true).mean()}

## 10. Smoke-test and resource preflight

The separate script selects smoke mode. In full mode this cell runs before any fit, measures staged/projected data, and stops early if RAM or disk is unsafe. Set `TRAINING_CV_FULL_PREFLIGHT_ONLY=1` to validate staging/resources without training.

In [ ]:
largest_model = max(MODEL_FEATURES, key=lambda name: len(MODEL_FEATURES[name]))
sample_dataset = ds.dataset(config.staging_dir / "cohort", format="parquet", partitioning="hive")
sample_columns = [TARGET] + MODEL_FEATURES[largest_model]
sample_table = sample_dataset.scanner(columns=sample_columns, batch_size=50_000).head(50_000)
sample_frame = sample_table.to_pandas()
sample_bytes_per_row = sample_frame.memory_usage(index=True, deep=True).sum() / max(1, len(sample_frame))
del sample_frame, sample_table
gc.collect()
vm = psutil.virtual_memory()
disk = psutil.disk_usage(ARTIFACT_DIR)
resource_report = estimate_full_resources(
    staging_bytes=staging_disk_bytes(config.staging_dir),
    row_count=manifest.row_count,
    largest_model_features=len(MODEL_FEATURES[largest_model]),
    sample_bytes_per_row=sample_bytes_per_row,
    available_ram_gb=vm.available / 1024**3,
    free_disk_gb=disk.free / 1024**3,
)
(ARTIFACT_DIR / f"resource_preflight{SUFFIX}.json").write_text(
    json.dumps(resource_report, indent=2, sort_keys=True), encoding="utf-8"
)
if config.run_mode == "full":
    assert config.run_mode == "full"
    assert_resources_safe(
        resource_report,
        min_available_ram_gb=config.min_available_ram_gb,
        min_free_disk_gb=config.min_free_disk_gb,
    )
display(pd.Series(resource_report, name="Resource preflight").to_frame("value"))
if FULL_PREFLIGHT_ONLY:
    print("Preflight-only requested: staging is valid and no model will be fitted.")

## 11. Full 10-fold execution

Models and folds run sequentially. Each completed held-out fold is atomically checkpointed with source, fold, staging, feature, model, and row-identity signatures. Valid checkpoints are reused after interruption.

In [ ]:
def fit_predict_fold(model_name, train_frame, valid_frame, fold_id):
    features = MODEL_FEATURES[model_name]
    x_train = train_frame[features]
    y_train = train_frame[TARGET].astype("int8")
    x_valid = valid_frame[features]
    y_valid = valid_frame[TARGET].astype("int8")
    extra = {}
    started = time.perf_counter()
    with threadpool_limits(limits=config.threads):
        if model_name == "logistic_structured":
            fitted = logistic_structured_pipeline().fit(x_train, y_train)
        elif model_name == "logistic_structured_nlp":
            inner_cv = StratifiedKFold(
                n_splits=N_INNER_FOLDS, shuffle=True, random_state=RANDOM_SEED
            )
            n_fit = len(x_train) * (N_INNER_FOLDS - 1) / N_INNER_FOLDS
            c_values = [1.0 / (penalty * n_fit) for penalty in LAMBDAS]
            search = GridSearchCV(
                logistic_nlp_pipeline(), {"model__C": c_values}, cv=inner_cv,
                scoring="neg_log_loss", refit=True, n_jobs=1,
            ).fit(x_train, y_train)
            fitted = search
            extra["best_C"] = float(search.best_params_["model__C"])
            extra["best_lambda"] = float(LAMBDAS[c_values.index(search.best_params_["model__C"])])
        elif model_name == "lightgbm_structured":
            prep = lightgbm_preprocessor(features)
            x_train_t = prep.fit_transform(x_train)
            estimator = lgb.LGBMClassifier(
                objective="binary", n_estimators=200, learning_rate=0.05,
                num_leaves=15, max_depth=5, min_child_samples=30,
                colsample_bytree=0.8, subsample=0.8, subsample_freq=1,
                random_state=RANDOM_SEED, n_jobs=config.threads, verbosity=-1,
            ).fit(x_train_t, y_train)
            fitted = (prep, estimator)
            del x_train_t
        else:
            x_inner_train, x_inner_eval, y_inner_train, y_inner_eval = train_test_split(
                x_train, y_train, test_size=0.10, stratify=y_train, random_state=RANDOM_SEED
            )
            tuning_prep = lightgbm_preprocessor(features)
            x_inner_train_t = tuning_prep.fit_transform(x_inner_train)
            x_inner_eval_t = tuning_prep.transform(x_inner_eval)
            tuning_estimator = lgb.LGBMClassifier(
                objective="binary", n_estimators=1000, learning_rate=0.05,
                num_leaves=31, min_child_samples=500, subsample=0.8,
                subsample_freq=1, colsample_bytree=0.8, reg_alpha=0.1,
                reg_lambda=1.0, random_state=RANDOM_SEED,
                n_jobs=config.threads, verbosity=-1,
            )
            tuning_estimator.fit(
                x_inner_train_t, y_inner_train,
                eval_X=x_inner_eval_t, eval_y=y_inner_eval,
                eval_metric=["auc", "binary_logloss"],
                callbacks=[lgb.early_stopping(50, verbose=False)],
            )
            best_iteration = max(1, int(tuning_estimator.best_iteration_))
            prep = lightgbm_preprocessor(features)
            x_train_t = prep.fit_transform(x_train)
            estimator = lgb.LGBMClassifier(
                objective="binary", n_estimators=best_iteration, learning_rate=0.05,
                num_leaves=31, min_child_samples=500, subsample=0.8,
                subsample_freq=1, colsample_bytree=0.8, reg_alpha=0.1,
                reg_lambda=1.0, random_state=RANDOM_SEED,
                n_jobs=config.threads, verbosity=-1,
            ).fit(x_train_t, y_train)
            fitted = (prep, estimator)
            extra.update({"best_iteration": best_iteration, "validation_strategy": "inner_holdout_refit"})
            del x_inner_train, x_inner_eval, y_inner_train, y_inner_eval
            del x_inner_train_t, x_inner_eval_t, tuning_estimator, tuning_prep, x_train_t
    training_seconds = time.perf_counter() - started
    prediction_started = time.perf_counter()
    if isinstance(fitted, tuple):
        prep, estimator = fitted
        probability = estimator.predict_proba(prep.transform(x_valid))[:, 1]
    else:
        probability = fitted.predict_proba(x_valid)[:, 1]
    prediction_seconds = time.perf_counter() - prediction_started
    return probability.astype("float32"), {
        **extra,
        "training_seconds": training_seconds,
        "prediction_seconds": prediction_seconds,
    }

def run_oof_model(model_name):
    prediction_parts = []
    metric_rows = []
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    for fold_id in range(N_OUTER_FOLDS):
        train_frame, valid_frame = load_model_fold_data(
            config.staging_dir, MODEL_FEATURES[model_name], fold_id, N_OUTER_FOLDS
        )
        expected = expected_checkpoint_signature(
            model_name=model_name,
            fold_id=fold_id,
            source_sha256=manifest.source_sha256,
            fold_assignment_sha256=manifest.fold_assignment_sha256,
            staging_signature=manifest.signature,
            feature_names=MODEL_FEATURES[model_name],
            model_parameters=MODEL_PARAMETERS[model_name],
            train_rows=len(train_frame),
            validation_stable_ids=valid_frame["stable_row_id"].to_numpy(),
        )
        checkpoint_path = CHECKPOINT_DIR / f"{model_name}_fold_{fold_id}.parquet"
        metadata_path = CHECKPOINT_DIR / f"{model_name}_fold_{fold_id}.json"
        if validate_checkpoint(checkpoint_path, metadata_path, expected):
            fold_prediction = pd.read_parquet(checkpoint_path)
            metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        else:
            probability, runtime = fit_predict_fold(model_name, train_frame, valid_frame, fold_id)
            fold_prediction = valid_frame[
                ["stable_row_id", "recommendationid", "fold_id", TARGET, DATE_COLUMN]
            ].rename(columns={TARGET: "actual_label"})
            fold_prediction[f"prob_{model_name}"] = probability
            metadata = {**expected, **runtime}
            temp_checkpoint = checkpoint_path.with_name(checkpoint_path.stem + ".tmp.parquet")
            temp_metadata = metadata_path.with_name(metadata_path.stem + ".tmp.json")
            fold_prediction.to_parquet(temp_checkpoint, index=False)
            temp_metadata.write_text(json.dumps(metadata, indent=2, sort_keys=True), encoding="utf-8")
            os.replace(temp_checkpoint, checkpoint_path)
            os.replace(temp_metadata, metadata_path)
        probability = fold_prediction[f"prob_{model_name}"].to_numpy()
        scores = probability_metrics(fold_prediction["actual_label"], probability)
        metric_rows.append({
            "model": model_name,
            "model_family": MODEL_FAMILY[model_name],
            "contains_nlp": MODEL_HAS_NLP[model_name],
            "fold_id": fold_id,
            "rows": len(fold_prediction),
            "positive_rate": fold_prediction["actual_label"].mean(),
            "training_seconds": metadata["training_seconds"],
            "prediction_seconds": metadata["prediction_seconds"],
            **scores,
        })
        prediction_parts.append(fold_prediction[["stable_row_id", f"prob_{model_name}"]])
        del train_frame, valid_frame, fold_prediction
        gc.collect()
    return pd.concat(prediction_parts, ignore_index=True), pd.DataFrame(metric_rows)

if not FULL_PREFLIGHT_ONLY:
    fold_metrics_parts = []
    oof_predictions = fold_assignments.copy()
    for model_name in MODEL_FEATURES:
        prediction_part, metric_part = run_oof_model(model_name)
        oof_predictions = oof_predictions.merge(
            prediction_part, on="stable_row_id", how="left", validate="one_to_one"
        )
        fold_metrics_parts.append(metric_part)
        del prediction_part, metric_part
        gc.collect()
    fold_metrics = pd.concat(fold_metrics_parts, ignore_index=True)
    probability_columns = [f"prob_{model}" for model in MODEL_FEATURES]
    assert len(oof_predictions) == manifest.row_count
    assert not oof_predictions[probability_columns].isna().any().any()
    probability_array = oof_predictions[probability_columns].to_numpy()
    assert np.isfinite(probability_array).all()
    assert ((probability_array >= 0) & (probability_array <= 1)).all()
    oof_predictions.to_parquet(ARTIFACT_DIR / f"oof_predictions{SUFFIX}.parquet", index=False)
    fold_metrics.to_csv(ARTIFACT_DIR / f"fold_metrics{SUFFIX}.csv", index=False)
    display(fold_metrics)

## 12. Fold-level and pooled OOF metrics

AP is primary; ROC-AUC is secondary. Log Loss and Brier Score remain uncalibrated probability diagnostics.

In [ ]:
if not FULL_PREFLIGHT_ONLY:
    summary_rows, business_rows, gains_rows = [], [], []
    actual = oof_predictions["actual_label"].to_numpy()
    for model_name in MODEL_FEATURES:
        model_fold = fold_metrics.loc[fold_metrics["model"].eq(model_name)]
        probability = oof_predictions[f"prob_{model_name}"].to_numpy()
        pooled = probability_metrics(actual, probability)
        for metric in ["average_precision", "roc_auc", "log_loss", "brier_score"]:
            values = model_fold[metric]
            summary_rows.append({
                "model": model_name, "metric": metric,
                "fold_mean": values.mean(), "fold_std": values.std(ddof=1),
                "fold_min": values.min(), "fold_max": values.max(),
                "pooled_oof": pooled[metric],
                "total_training_seconds": model_fold["training_seconds"].sum(),
                "total_prediction_seconds": model_fold["prediction_seconds"].sum(),
            })
        top10 = top_fraction_metrics(actual, probability, 0.10)
        top20 = top_fraction_metrics(actual, probability, 0.20)
        business_rows.append({
            "model": model_name,
            "precision_at_10pct": top10["precision"],
            "precision_at_20pct": top20["precision"],
            "recall_at_20pct": top20["recall"],
            "lift_at_10pct": top10["lift"],
            "lift_at_20pct": top20["lift"],
        })
        order = np.argsort(-probability, kind="stable")
        cumulative_positive = np.cumsum(actual[order])
        for fraction in np.linspace(0.01, 1.0, 100):
            count = max(1, int(np.ceil(len(actual) * fraction)))
            gains_rows.append({
                "model": model_name,
                "population_fraction": fraction,
                "cumulative_gain": cumulative_positive[count - 1] / cumulative_positive[-1],
                "cumulative_lift": (cumulative_positive[count - 1] / count) / actual.mean(),
            })
    cv_summary = pd.DataFrame(summary_rows)
    business_metrics = pd.DataFrame(business_rows)
    cumulative_gains = pd.DataFrame(gains_rows)
    cv_summary.to_csv(ARTIFACT_DIR / f"cv_summary{SUFFIX}.csv", index=False)
    business_metrics.to_csv(ARTIFACT_DIR / f"business_metrics{SUFFIX}.csv", index=False)
    cumulative_gains.to_csv(ARTIFACT_DIR / f"cumulative_gains{SUFFIX}.csv", index=False)
    display(cv_summary)

## 13. OOF ROC and precision–recall curves

Curves use only held-out OOF probabilities.

In [ ]:
if not FULL_PREFLIGHT_ONLY:
    actual = oof_predictions["actual_label"].to_numpy()
    prevalence = actual.mean()
    fig, ax = plt.subplots(figsize=(8, 6))
    for model_name in MODEL_FEATURES:
        probability = oof_predictions[f"prob_{model_name}"].to_numpy()
        fpr, tpr, _ = roc_curve(actual, probability)
        ax.plot(fpr, tpr, label=f"{model_name} (AUC={roc_auc_score(actual, probability):.3f})")
    ax.plot([0, 1], [0, 1], "--", color="gray", label="Random")
    ax.set(title="Training-stage pooled OOF ROC curves", xlabel="False positive rate", ylabel="True positive rate")
    ax.legend(fontsize=8); fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"01_oof_roc_curves{SUFFIX}.png", dpi=180); plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 6))
    for model_name in MODEL_FEATURES:
        probability = oof_predictions[f"prob_{model_name}"].to_numpy()
        precision, recall, _ = precision_recall_curve(actual, probability)
        ax.plot(recall, precision, label=f"{model_name} (AP={average_precision_score(actual, probability):.3f})")
    ax.axhline(prevalence, linestyle="--", color="gray", label=f"Prevalence={prevalence:.3f}")
    ax.set(title="Training-stage pooled OOF precision–recall curves", xlabel="Recall", ylabel="Precision")
    ax.legend(fontsize=8); fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"02_oof_precision_recall_curves{SUFFIX}.png", dpi=180); plt.close(fig)

## 14. NLP incremental-value analysis

Shared-fold AP differences quantify the incremental contribution of NLP features.

In [ ]:
if not FULL_PREFLIGHT_ONLY:
    def fold_comparison(metric, filename):
        order = list(MODEL_FEATURES)
        summary = fold_metrics.groupby("model")[metric].agg(["mean", "std"]).reindex(order)
        fig, ax = plt.subplots(figsize=(10, 6))
        for position, model_name in enumerate(order):
            values = fold_metrics.loc[fold_metrics["model"].eq(model_name), metric]
            ax.scatter(np.full(len(values), position), values, s=25, alpha=0.7)
            ax.errorbar(position, summary.loc[model_name, "mean"], yerr=summary.loc[model_name, "std"], fmt="o", color="black", capsize=5)
        ax.set_xticks(range(len(order)), order, rotation=20, ha="right")
        ax.set_title(f"Training-stage {N_OUTER_FOLDS}-fold {metric.replace('_', ' ').title()}")
        fig.tight_layout(); fig.savefig(FIGURE_DIR / filename, dpi=180); plt.close(fig)
    fold_comparison("average_precision", f"03_fold_average_precision{SUFFIX}.png")
    fold_comparison("roc_auc", f"04_fold_roc_auc{SUFFIX}.png")

    fold_long = fold_metrics.melt(id_vars=["model", "fold_id"], value_vars=["average_precision", "roc_auc"], var_name="metric", value_name="score")
    fig, ax = plt.subplots(figsize=(11, 6))
    sns.boxplot(data=fold_long, x="model", y="score", hue="metric", ax=ax)
    sns.stripplot(data=fold_long, x="model", y="score", hue="metric", dodge=True, alpha=0.6, size=4, ax=ax, legend=False)
    ax.set_title("Training-stage fold-level score distributions"); ax.tick_params(axis="x", rotation=20)
    fig.tight_layout(); fig.savefig(FIGURE_DIR / f"05_fold_score_distributions{SUFFIX}.png", dpi=180); plt.close(fig)

    pivot_ap = fold_metrics.pivot(index="fold_id", columns="model", values="average_precision")
    nlp_increment = pd.DataFrame({
        "fold_id": pivot_ap.index,
        "Logistic Regression": pivot_ap["logistic_structured_nlp"] - pivot_ap["logistic_structured"],
        "LightGBM": pivot_ap["lightgbm_structured_nlp"] - pivot_ap["lightgbm_structured"],
    }).melt(id_vars="fold_id", var_name="model_family", value_name="ap_increment")
    nlp_increment.to_csv(ARTIFACT_DIR / f"nlp_incremental_value{SUFFIX}.csv", index=False)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.stripplot(data=nlp_increment, x="model_family", y="ap_increment", jitter=0.08, ax=ax)
    means = nlp_increment.groupby("model_family")["ap_increment"].mean()
    for position, family in enumerate(means.index): ax.scatter(position, means.loc[family], s=100, marker="D", color="black")
    ax.axhline(0, linestyle="--", color="gray"); ax.set_title("Training-stage fold-level NLP incremental AP")
    fig.tight_layout(); fig.savefig(FIGURE_DIR / f"06_nlp_incremental_value{SUFFIX}.png", dpi=180); plt.close(fig)

## 15. Business-ranking metrics

Precision, recall, cumulative gains, and lift describe prioritization without selecting a classification threshold.

In [ ]:
if not FULL_PREFLIGHT_ONLY:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for model_name in MODEL_FEATURES:
        curve = cumulative_gains.loc[cumulative_gains["model"].eq(model_name)]
        axes[0].plot(curve["population_fraction"], curve["cumulative_gain"], label=model_name)
        axes[1].plot(curve["population_fraction"], curve["cumulative_lift"], label=model_name)
    axes[0].plot([0, 1], [0, 1], "--", color="gray")
    axes[0].set(title="Training-stage OOF cumulative gains", xlabel="Population fraction", ylabel="Captured positives")
    axes[1].axhline(1, linestyle="--", color="gray")
    axes[1].set(title="Training-stage OOF cumulative lift", xlabel="Population fraction", ylabel="Lift")
    for ax in axes: ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(FIGURE_DIR / f"07_oof_gains_lift{SUFFIX}.png", dpi=180); plt.close(fig)
    display(business_metrics)

## graphs

In [ ]:
# ============================================================
# Final presentation figures from completed full OOF artifacts
# This cell does NOT retrain any model.
# ============================================================

import gc
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

# ARTIFACT_DIR must come from the centralized configuration cell.
if "ARTIFACT_DIR" not in globals():
    raise RuntimeError(
        "ARTIFACT_DIR is not defined. Run the configuration cell first."
    )

artifact_dir = Path(ARTIFACT_DIR)
figure_dir = artifact_dir / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)


def resolve_artifact(*filenames):
    """Return the first existing artifact from the supplied candidates."""
    for filename in filenames:
        candidate = artifact_dir / filename
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "None of the expected artifacts exists:\n"
        + "\n".join(str(artifact_dir / name) for name in filenames)
    )


# Current Notebook uses names without "_full".
# The Windows version may use explicit "_full" names.
fold_metrics_path = resolve_artifact(
    "fold_metrics.csv",
    "fold_metrics_full.csv",
)

oof_predictions_path = resolve_artifact(
    "oof_predictions.parquet",
    "oof_predictions_full.parquet",
)

print("Fold metrics:", fold_metrics_path)
print("OOF predictions:", oof_predictions_path)

MODEL_ORDER = [
    "logistic_structured",
    "logistic_structured_nlp",
    "lightgbm_structured",
    "lightgbm_structured_nlp",
]

MODEL_LABELS = {
    "logistic_structured": "Logistic\nStructured",
    "logistic_structured_nlp": "L2 Logistic\nStructured + NLP",
    "lightgbm_structured": "LightGBM\nStructured",
    "lightgbm_structured_nlp": "LightGBM\nStructured + NLP",
}

MODEL_LEGEND_LABELS = {
    "logistic_structured": "Logistic — Structured",
    "logistic_structured_nlp": "L2 Logistic — Structured + NLP",
    "lightgbm_structured": "LightGBM — Structured",
    "lightgbm_structured_nlp": "LightGBM — Structured + NLP",
}

MODEL_COLORS = {
    "logistic_structured": "#7F9FA0",
    "logistic_structured_nlp": "#216B95",
    "lightgbm_structured": "#DF8C44",
    "lightgbm_structured_nlp": "#A63D20",
}

# ============================================================
# Validate full 10-fold metrics
# ============================================================

fold_plot = pd.read_csv(fold_metrics_path)

required_fold_columns = {
    "model",
    "fold_id",
    "average_precision",
    "roc_auc",
}
missing_fold_columns = required_fold_columns - set(fold_plot.columns)

assert not missing_fold_columns, (
    f"Missing fold-metric columns: {sorted(missing_fold_columns)}"
)

assert set(fold_plot["model"]) == set(MODEL_ORDER), (
    "Fold metrics do not contain exactly the four expected models."
)

fold_counts = (
    fold_plot.groupby("model")["fold_id"]
    .nunique()
    .reindex(MODEL_ORDER)
)

assert fold_counts.eq(10).all(), (
    f"Expected 10 folds per model, found:\n{fold_counts}"
)

assert len(fold_plot) == 40, (
    f"Expected 40 model-fold rows, found {len(fold_plot):,}."
)

# ============================================================
# Figure 1: three-panel 10-fold model comparison
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(20, 8.5),
    gridspec_kw={"width_ratios": [1.15, 1.15, 0.90]},
)

metric_specs = [
    (
        "average_precision",
        "A. PR-AUC / Average Precision",
        "Average precision",
    ),
    (
        "roc_auc",
        "B. ROC-AUC",
        "ROC-AUC",
    ),
]

for ax, (metric, title, ylabel) in zip(axes[:2], metric_specs):
    all_values = fold_plot[metric].to_numpy()
    metric_range = max(
        float(all_values.max() - all_values.min()),
        0.001,
    )

    for position, model_name in enumerate(MODEL_ORDER):
        values = (
            fold_plot.loc[
                fold_plot["model"].eq(model_name),
                ["fold_id", metric],
            ]
            .sort_values("fold_id")[metric]
            .to_numpy()
        )

        mean_value = float(values.mean())
        std_value = float(values.std(ddof=1))

        # Deterministic horizontal offsets for the ten folds.
        offsets = np.linspace(-0.075, 0.075, len(values))

        ax.scatter(
            position + offsets,
            values,
            s=34,
            alpha=0.65,
            color=MODEL_COLORS[model_name],
            edgecolor="none",
            zorder=2,
        )

        ax.errorbar(
            position,
            mean_value,
            yerr=std_value,
            fmt="o",
            markersize=11,
            color=MODEL_COLORS[model_name],
            markeredgecolor="black",
            markeredgewidth=0.8,
            ecolor="#333333",
            elinewidth=1.5,
            capsize=5,
            zorder=3,
        )

        ax.text(
            position,
            mean_value + std_value + 0.035 * metric_range,
            f"{mean_value:.4f}",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
        )

    ax.set_xticks(
        range(len(MODEL_ORDER)),
        [MODEL_LABELS[name] for name in MODEL_ORDER],
    )
    ax.set_title(title, loc="left", fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.grid(axis="x", visible=False)
    ax.margins(x=0.08, y=0.18)

# Paired fold-level NLP AP improvements.
ap_pivot = (
    fold_plot.pivot(
        index="fold_id",
        columns="model",
        values="average_precision",
    )
    .sort_index()
)

nlp_uplift = {
    "Logistic": (
        ap_pivot["logistic_structured_nlp"]
        - ap_pivot["logistic_structured"]
    ),
    "LightGBM": (
        ap_pivot["lightgbm_structured_nlp"]
        - ap_pivot["lightgbm_structured"]
    ),
}

uplift_colors = {
    "Logistic": MODEL_COLORS["logistic_structured_nlp"],
    "LightGBM": MODEL_COLORS["lightgbm_structured_nlp"],
}

uplift_ax = axes[2]

for position, family in enumerate(["Logistic", "LightGBM"]):
    values = nlp_uplift[family].to_numpy()
    mean_value = float(values.mean())
    std_value = float(values.std(ddof=1))
    offsets = np.linspace(-0.055, 0.055, len(values))

    uplift_ax.scatter(
        position + offsets,
        values,
        s=38,
        alpha=0.70,
        color=uplift_colors[family],
        edgecolor="none",
        zorder=2,
    )

    uplift_ax.errorbar(
        position,
        mean_value,
        yerr=std_value,
        fmt="D",
        markersize=10,
        color=uplift_colors[family],
        markeredgecolor="black",
        markeredgewidth=0.8,
        ecolor="#333333",
        elinewidth=1.5,
        capsize=5,
        zorder=3,
    )

    uplift_ax.annotate(
        f"{mean_value:+.4f}",
        xy=(position, mean_value),
        xytext=(0, 18),
        textcoords="offset points",
        ha="center",
        fontsize=11,
        fontweight="bold",
    )

uplift_ax.axhline(
    0,
    linestyle="--",
    linewidth=1.2,
    color="gray",
)

uplift_ax.set_xticks(
    [0, 1],
    ["Logistic\nNLP uplift", "LightGBM\nNLP uplift"],
)
uplift_ax.set_title(
    "C. Incremental Value of NLP",
    loc="left",
    fontweight="bold",
)
uplift_ax.set_ylabel("Paired fold Δ average precision")
uplift_ax.grid(axis="x", visible=False)
uplift_ax.margins(x=0.25, y=0.25)

fig.suptitle(
    "Training-Stage 10-Fold OOF Model Comparison (2010–2021)",
    fontsize=17,
    fontweight="bold",
    y=0.97,
)

fig.tight_layout(rect=[0, 0.07, 1, 0.92])

comparison_path = (
    figure_dir / "08_10fold_model_comparison_full.png"
)

fig.savefig(
    comparison_path,
    dpi=220,
    bbox_inches="tight",
)
plt.close(fig)

# ============================================================
# Load only columns required for pooled OOF curves
# ============================================================

probability_columns = [
    f"prob_{model_name}"
    for model_name in MODEL_ORDER
]

oof_columns = [
    "stable_row_id",
    "actual_label",
    *probability_columns,
]

oof_plot = pd.read_parquet(
    oof_predictions_path,
    columns=oof_columns,
)

expected_rows = int(
    fold_plot.loc[
        fold_plot["model"].eq(MODEL_ORDER[0]),
        "rows",
    ].sum()
)

assert len(oof_plot) == expected_rows, (
    f"OOF row count mismatch: "
    f"{len(oof_plot):,} vs expected {expected_rows:,}."
)

assert oof_plot["stable_row_id"].is_unique, (
    "stable_row_id is not unique in pooled OOF predictions."
)

assert oof_plot["actual_label"].isin([0, 1]).all(), (
    "OOF actual_label contains values other than 0 and 1."
)

probability_array = oof_plot[
    probability_columns
].to_numpy(copy=False)

assert np.isfinite(probability_array).all(), (
    "OOF probabilities contain NaN or infinite values."
)

assert (
    (probability_array >= 0)
    & (probability_array <= 1)
).all(), "OOF probabilities are outside [0, 1]."

actual = oof_plot["actual_label"].to_numpy(dtype=np.int8)
prevalence = float(actual.mean())


def thin_curve(x, y, max_points=25_000):
    """
    Retain endpoints and evenly spaced curve points for fast rendering.
    Metrics are still calculated from all OOF observations.
    """
    if len(x) <= max_points:
        return x, y

    indices = np.linspace(
        0,
        len(x) - 1,
        max_points,
        dtype=np.int64,
    )
    indices = np.unique(indices)

    return x[indices], y[indices]


# ============================================================
# Figure 2: pooled OOF ROC and precision–recall curves
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(17, 7))

roc_ax, pr_ax = axes

for model_name in MODEL_ORDER:
    probability = oof_plot[
        f"prob_{model_name}"
    ].to_numpy(dtype=np.float32, copy=False)

    # ROC curve and exact pooled ROC-AUC.
    fpr, tpr, _ = roc_curve(
        actual,
        probability,
        drop_intermediate=True,
    )

    roc_auc = roc_auc_score(
        actual,
        probability,
    )

    fpr_plot, tpr_plot = thin_curve(fpr, tpr)

    roc_ax.plot(
        fpr_plot,
        tpr_plot,
        linewidth=2.2,
        color=MODEL_COLORS[model_name],
        label=(
            f"{MODEL_LEGEND_LABELS[model_name]} "
            f"(AUC={roc_auc:.4f})"
        ),
    )

    del fpr, tpr, fpr_plot, tpr_plot

    # PR curve and exact pooled average precision.
    precision, recall, _ = precision_recall_curve(
        actual,
        probability,
    )

    average_precision = average_precision_score(
        actual,
        probability,
    )

    recall_plot, precision_plot = thin_curve(
        recall,
        precision,
    )

    pr_ax.plot(
        recall_plot,
        precision_plot,
        linewidth=2.2,
        color=MODEL_COLORS[model_name],
        label=(
            f"{MODEL_LEGEND_LABELS[model_name]} "
            f"(AP={average_precision:.4f})"
        ),
    )

    del precision, recall, recall_plot, precision_plot
    gc.collect()

roc_ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5,
    color="gray",
    label="Random baseline",
)

roc_ax.set(
    title="A. Pooled OOF ROC Curves",
    xlabel="False positive rate",
    ylabel="True positive rate",
    xlim=(0, 1),
    ylim=(0, 1),
)

roc_ax.legend(
    loc="lower right",
    fontsize=9,
)

pr_ax.axhline(
    prevalence,
    linestyle="--",
    linewidth=1.5,
    color="gray",
    label=f"Random baseline ({prevalence:.4f})",
)

pr_ax.set(
    title="B. Pooled OOF Precision–Recall Curves",
    xlabel="Recall",
    ylabel="Precision",
    xlim=(0, 1),
    ylim=(0, 1),
)

pr_ax.legend(
    loc="lower left",
    fontsize=9,
)

for ax in axes:
    ax.grid(alpha=0.30)

fig.suptitle(
    "Training-Stage Pooled Out-of-Fold Performance (2010–2021)",
    fontsize=17,
    fontweight="bold",
    y=1.01,
)

fig.tight_layout(rect=[0, 0.035, 1, 0.96])

curves_path = (
    figure_dir / "09_pooled_oof_roc_pr_full.png"
)

fig.savefig(
    curves_path,
    dpi=220,
    bbox_inches="tight",
)
plt.close(fig)

# Release the large OOF frame before displaying images.
del probability_array, oof_plot
gc.collect()

print(f"Saved: {comparison_path}")
print(f"Saved: {curves_path}")

display(Image(filename=str(comparison_path)))
display(Image(filename=str(curves_path)))

## 16. Optional full-training refits

Refits are skipped by default. To create them after a complete full OOF run, set `TRAINING_CV_RUN_FULL_REFITS=1` and rerun; valid fold checkpoints are reused before the four all-training fits begin.

In [ ]:
def fit_full_candidate(model_name, data):
    features = MODEL_FEATURES[model_name]
    x, y = data[features], data[TARGET].astype("int8")
    if model_name == "logistic_structured":
        return logistic_structured_pipeline().fit(x, y), {}
    if model_name == "logistic_structured_nlp":
        inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
        n_fit = len(x) * 2 / 3
        values = [1.0 / (penalty * n_fit) for penalty in LAMBDAS]
        search = GridSearchCV(logistic_nlp_pipeline(), {"model__C": values}, cv=inner, scoring="neg_log_loss", refit=True, n_jobs=1).fit(x, y)
        return search.best_estimator_, {"best_C": search.best_params_["model__C"]}
    prep = lightgbm_preprocessor(features)
    transformed = prep.fit_transform(x)
    if model_name == "lightgbm_structured":
        estimator = lgb.LGBMClassifier(objective="binary", n_estimators=200, learning_rate=0.05, num_leaves=15, max_depth=5, min_child_samples=30, colsample_bytree=0.8, subsample=0.8, subsample_freq=1, random_state=RANDOM_SEED, n_jobs=config.threads, verbosity=-1)
    else:
        iterations = [json.loads(path.read_text()).get("best_iteration") for path in CHECKPOINT_DIR.glob("lightgbm_structured_nlp_fold_*.json")]
        iterations = [value for value in iterations if value]
        estimator = lgb.LGBMClassifier(objective="binary", n_estimators=int(np.median(iterations)), learning_rate=0.05, num_leaves=31, min_child_samples=500, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=RANDOM_SEED, n_jobs=config.threads, verbosity=-1)
    estimator.fit(transformed, y)
    return (prep, estimator), {"n_estimators": estimator.n_estimators}

if not FULL_PREFLIGHT_ONLY and RUN_FULL_REFITS:
    assert RUN_MODE == "full"
    assert len(list(CHECKPOINT_DIR.glob("*.parquet"))) == len(MODEL_FEATURES) * N_OUTER_FOLDS
    FITTED_DIR.mkdir(parents=True, exist_ok=True)
    fitted_rows = []
    all_dataset = ds.dataset(config.staging_dir / "cohort", format="parquet", partitioning="hive")
    for model_name, features in MODEL_FEATURES.items():
        columns = [TARGET] + features
        all_data = all_dataset.to_table(columns=columns).to_pandas()
        fitted, extra = fit_full_candidate(model_name, all_data)
        output = FITTED_DIR / f"{model_name}.joblib"
        joblib.dump({"model": fitted, "features": features, "rows": len(all_data), "seed": RANDOM_SEED, **extra}, output)
        fitted_rows.append({"model": model_name, "path": str(output), **extra})
        del all_data, fitted
        gc.collect()
    pd.DataFrame(fitted_rows).to_csv(FITTED_DIR / "fitted_model_manifest.csv", index=False)
else:
    print("Full refits disabled (default).")

## 17. Limitations

Cross-validation is random review-level validation, not temporal validation. The target remains an observed engagement proxy. Full execution time depends strongly on Logistic NLP inner-CV cost. Do not describe results as final model performance until later temporal validation.

## 18. Artifact manifest and final verification

This cell verifies protected inputs again, labels the leading candidate only at the CV stage, and records run metadata. Staging partitions are summarized by their signed manifest rather than individually hashed.

In [ ]:
source_hash_after = sha256_file(config.source_path)
assert source_hash_after == source_hash_before
for row in protected_hashes.itertuples(index=False):
    assert sha256_file(Path(row.path)) == row.sha256

if not FULL_PREFLIGHT_ONLY:
    ap = cv_summary.loc[cv_summary["metric"].eq("average_precision"), ["model", "fold_mean", "fold_std", "pooled_oof"]].rename(columns={"fold_mean": "average_precision_mean", "fold_std": "average_precision_std", "pooled_oof": "average_precision_pooled_oof"})
    roc = cv_summary.loc[cv_summary["metric"].eq("roc_auc"), ["model", "fold_mean", "fold_std"]].rename(columns={"fold_mean": "roc_auc_mean", "fold_std": "roc_auc_std"})
    ranking = ap.merge(roc, on="model").sort_values(["average_precision_mean", "roc_auc_mean"], ascending=False).reset_index(drop=True)
    ranking["status"] = "CV-stage candidate"
    ranking.loc[0, "status"] = "CV-stage leading candidate"
    ranking.to_csv(ARTIFACT_DIR / f"model_ranking{SUFFIX}.csv", index=False)
    display(ranking)

run_manifest = {
    "run_mode": RUN_MODE,
    "full_run_completed": bool(RUN_MODE == "full" and not FULL_PREFLIGHT_ONLY),
    "full_refits_completed": bool(RUN_MODE == "full" and RUN_FULL_REFITS and not FULL_PREFLIGHT_ONLY),
    "rows": manifest.row_count,
    "positive_rows": manifest.positive_rows,
    "folds": N_OUTER_FOLDS,
    "models": list(MODEL_FEATURES),
    "source_sha256": manifest.source_sha256,
    "fold_assignment_sha256": manifest.fold_assignment_sha256,
    "staging_signature": manifest.signature,
    "peak_rss_gb": peak_rss_gb(),
    "elapsed_seconds": time.perf_counter() - run_started,
    "package_versions": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__, "sklearn": sklearn.__version__, "lightgbm": lgb.__version__},
}
(ARTIFACT_DIR / f"run_manifest{SUFFIX}.json").write_text(json.dumps(run_manifest, indent=2, sort_keys=True), encoding="utf-8")

artifact_rows = []
for path in sorted(ARTIFACT_DIR.rglob("*")):
    relative = path.relative_to(ARTIFACT_DIR)
    if not path.is_file() or any(part.startswith(".") for part in relative.parts):
        continue
    if "staging" in relative.parts or path.name == "artifact_manifest.csv":
        continue
    artifact_rows.append({"relative_path": str(relative), "bytes": path.stat().st_size, "sha256": sha256_file(path)})
pd.DataFrame(artifact_rows).to_csv(ARTIFACT_DIR / "artifact_manifest.csv", index=False)
display(pd.Series(run_manifest, name="Run manifest").to_frame("value"))